#### Spacy Linguistic Annotations and Word vectors

##### Linguistic Features

Word-sense disambiguation with spaCy

In [1]:
import spacy

In [2]:
nlp = spacy.load("en_core_web_sm")
nlp_md = spacy.load("en_core_web_md")

In [3]:
texts = ["This device is used to jam the signal.",
         "I am stuck in a traffic jam"]

docs = [nlp(text) for text in texts]

for i, doc in enumerate(docs):
    print(f"Sentence {i+1} : {[(token.text, token.pos_) for token in doc if "jam" in token.text]}")

Sentence 1 : [('jam', 'VERB')]
Sentence 2 : [('jam', 'NOUN')]


In [4]:
text = "This device is used to jam the signal."

doc = nlp(text)

[print(token.text, token.pos_) for token in doc if "jam" in token.text]

jam VERB


[None]

Dependency Parsing with Spacy

In [5]:
texts = ['I want to fly from Boston at 8:38 am and arrive in Denver at 11:10 in the morning',
 'What flights are available from Pittsburgh to Baltimore on Thursday morning?',
 'What is the arrival time in San francisco for the 7:55 AM flight leaving Washington?']

docs = [nlp(text) for text in texts]

for doc in docs[:1]:
    [print(token.text, token.dep_, spacy.explain(token.dep_)) for token in doc]

I nsubj nominal subject
want ROOT root
to aux auxiliary
fly xcomp open clausal complement
from prep prepositional modifier
Boston pobj object of preposition
at prep prepositional modifier
8:38 nummod numeric modifier
am pobj object of preposition
and cc coordinating conjunction
arrive conj conjunct
in prep prepositional modifier
Denver pobj object of preposition
at prep prepositional modifier
11:10 pobj object of preposition
in prep prepositional modifier
the det determiner
morning pobj object of preposition


##### Introduction to Word Vectors

Spacy Vocabulary

In [7]:
nlp_md.meta

{'lang': 'en',
 'name': 'core_web_md',
 'version': '3.8.0',
 'description': 'English pipeline optimized for CPU. Components: tok2vec, tagger, parser, senter, ner, attribute_ruler, lemmatizer.',
 'author': 'Explosion',
 'email': 'contact@explosion.ai',
 'url': 'https://explosion.ai',
 'license': 'MIT',
 'spacy_version': '>=3.8.0,<3.9.0',
 'spacy_git_version': '5010fcbd3',
 'vectors': {'width': 300,
  'vectors': 20000,
  'keys': 684830,
  'name': 'en_vectors',
  'mode': 'default'},
 'labels': {'tok2vec': [],
  'tagger': ['$',
   "''",
   ',',
   '-LRB-',
   '-RRB-',
   '.',
   ':',
   'ADD',
   'AFX',
   'CC',
   'CD',
   'DT',
   'EX',
   'FW',
   'HYPH',
   'IN',
   'JJ',
   'JJR',
   'JJS',
   'LS',
   'MD',
   'NFP',
   'NN',
   'NNP',
   'NNPS',
   'NNS',
   'PDT',
   'POS',
   'PRP',
   'PRP$',
   'RB',
   'RBR',
   'RBS',
   'RP',
   'SYM',
   'TO',
   'UH',
   'VB',
   'VBD',
   'VBG',
   'VBN',
   'VBP',
   'VBZ',
   'WDT',
   'WP',
   'WP$',
   'WRB',
   'XX',
   '_SP',
   '``'

In [9]:
print(nlp_md.meta["vectors"])

{'width': 300, 'vectors': 20000, 'keys': 684830, 'name': 'en_vectors', 'mode': 'default'}


In [11]:
# Print the number of words in the model's vocabulary
nlp_md.meta["vectors"]["vectors"]

20000

In [ ]:
# Print the dimensions of word vectors in en_core_web_md model
nlp_md.meta["vectors"]["width"]

300

Word Vector in Spacy Vocabulary

In [20]:
words = ["like", "love"]

# ID's for all the given words
ids = [nlp_md.vocab.strings[word] for word in words]
print("ID's: ", ids)

# store first 10 elements of the word vectors for each word
word_vectors = [nlp_md.vocab.vectors[word][:10] for word in words]
print("Word Vectors: ", word_vectors)

# Print the first ten elements of the first word vector
print(word_vectors[0])

ID's:  [18194338103975822726, 3702023516439754181]
Word Vectors:  [array([-0.61052 ,  0.11656 , -0.50648 , -0.32216 , -0.099742,  0.10182 ,
        0.31042 , -0.18155 ,  0.31774 ,  2.1537  ], dtype=float32), array([-0.61052 ,  0.11656 , -0.50648 , -0.32216 , -0.099742,  0.10182 ,
        0.31042 , -0.18155 ,  0.31774 ,  2.1537  ], dtype=float32)]
[-0.61052   0.11656  -0.50648  -0.32216  -0.099742  0.10182   0.31042
 -0.18155   0.31774   2.1537  ]


Word Vectors and Spacy

In [22]:
from sklearn.decomposition import PCA
import numpy as np 

pca = PCA(n_components=2)

words = ["tiger", "bird"]

# extract the id's of the given words
word_ids = [nlp_md.vocab.strings[word] for word in words]

word_vectors = np.vstack([nlp_md.vocab.vectors[i][:5] for i in word_ids])
word_vectors_transformed = pca.fit_transform(word_vectors)

print(word_vectors_transformed[:, 0])

[-0.47121444  0.47121444]


Most Similar Words

In [26]:
word = "computer"
word_vector = nlp_md.vocab.vectors[word]

In [37]:
most_similar_words = nlp_md.vocab.vectors.most_similar(np.asarray([word_vector]), n=1)
words = [nlp.vocab.strings[w] for w in most_similar_words[0][0]]
print(words)

['Paths']


Measuring Semantic Similarity

Doc similarity with spaCy

In [39]:
texts = ['I like the Vitality canned dog food products.',
 'The peanuts were actually small sized unsalted. Not sure if this was an error.',
 'It is a light, pillowy citrus gelatin with nuts - in this case Filberts.',
 'the Root Beer Extract I ordered is very medicinal.',
 'Great taffy at a great price.']

category = "canned dog food"

docs = [nlp_md(text) for text in texts]
category_document = nlp_md(category)

for doc in docs:
    print(doc, "<->", category_document, doc.similarity(category_document))
    print()

I like the Vitality canned dog food products. <-> canned dog food 0.7619024515151978

The peanuts were actually small sized unsalted. Not sure if this was an error. <-> canned dog food 0.4949886202812195

It is a light, pillowy citrus gelatin with nuts - in this case Filberts. <-> canned dog food 0.571582019329071

the Root Beer Extract I ordered is very medicinal. <-> canned dog food 0.5482636094093323

Great taffy at a great price. <-> canned dog food 0.3602704107761383



Span Similarity with spacym

In [41]:
category = "canned dog food"
category_document = nlp_md(category)

document = nlp_md("canned food products")

document_span = document[0:3]
print(f"Semantic similarity with", document_span.text, ":", round(document_span.similarity(category_document), 3))

Semantic similarity with canned food products : 0.8


Semantic Similarity for categorizing Text

In [44]:
text = 'This hot sauce is amazing! We picked up a bottle on a trip!'
key = nlp_md("sauce")
sentences = nlp_md(text)

for sent in sentences.sents:
    print("Similarity Score:", round(sent.similarity(key), 3))

Similarity Score: 0.593
Similarity Score: 0.476
